# Fine-tune Longformer Encoder-Decoder on OpenNyAI-tagged judgments 

This version trains LED with the **1,200 tagged training judgments**. It makes a deterministic 90/10 train/validation split from those 1,200 (about 1,080/120). The separate **100 tagged test judgments** are kept out of training and validation and used only for final evaluation against their original reference summaries.



## 1. Install/check dependenciesKaggle usually includes PyTorch. This installs the model/data packages without reinstalling PyTorch. If Kaggle reports a kernel restart is needed, restart once and run from the top.

In [ ]:
%pip install -q "transformers>=4.44,<5" "datasets>=2.20,<4" "peft>=0.12,<1" accelerate sentencepiece rouge-score

import torch, transformers, datasets, peft
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. In Kaggle: Settings → Accelerator → GPU, then rerun.')

## 2. Locate the tagged train/test files and original IN-Abs pairs


In [ ]:
from pathlib import Path
import json, random, re, os

INPUT_ROOT = Path('/kaggle/input')
TRAIN_TAGGED_HINT = 'train-1200'   # Kaggle input folder containing the 1,200 train tags
TEST_TAGGED_HINT = 'test-100' # Kaggle input folder containing the 100 test tags; change if needed
ORIGINAL_DATA_HINT = 'legallensdataset' # folder containing original train.jsonl and test.jsonl
TRAIN_ROLES_CSV = None
TEST_ROLES_CSV = None

def find_by_hint(filename, hint):
    hits = [p for p in INPUT_ROOT.rglob(filename) if hint.lower() in str(p).lower()]
    if len(hits) != 1:
        raise FileNotFoundError(
            f'Expected exactly one {filename} under a path containing {hint!r}; found {len(hits)}: {hits}. '
            'Update the folder hint in this cell.'
        )
    return hits[0]

TRAIN_TAGGED_JSONL = find_by_hint('tagged.jsonl', TRAIN_TAGGED_HINT)
TEST_TAGGED_JSONL = find_by_hint('tagged.jsonl', TEST_TAGGED_HINT)
TRAIN_JSONL = find_by_hint('train.jsonl', ORIGINAL_DATA_HINT)
TEST_JSONL = find_by_hint('test.jsonl', ORIGINAL_DATA_HINT)

train_role_hits = [p for p in INPUT_ROOT.rglob('sentence_roles.csv') if TRAIN_TAGGED_HINT.lower() in str(p).lower()]
test_role_hits = [p for p in INPUT_ROOT.rglob('sentence_roles.csv') if TEST_TAGGED_HINT.lower() in str(p).lower()]
TRAIN_ROLES_CSV = train_role_hits[0] if train_role_hits else None
TEST_ROLES_CSV = test_role_hits[0] if test_role_hits else None

for label, path in [('Train tags', TRAIN_TAGGED_JSONL), ('Test tags', TEST_TAGGED_JSONL),
                    ('Original train', TRAIN_JSONL), ('Original test', TEST_JSONL),
                    ('Train roles CSV', TRAIN_ROLES_CSV), ('Test roles CSV', TEST_ROLES_CSV)]:
    print(f'{label}: {path}')

## 3. Match tags to the original judgments and reference summaries


In [ ]:
import csv

def norm_id(value):
    if value is None:
        return None
    return Path(str(value).replace('\\', '/')).stem.strip()

def first_value(row, keys):
    for key in keys:
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return None

def read_original_jsonl(path):
    result = {}
    with path.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            case_id = norm_id(row.get('id') or row.get('doc_id') or row.get('source_file'))
            judgment = first_value(row, ['judgment', 'judgement', 'document', 'text'])
            summary = first_value(row, ['summary', 'reference_summary', 'target'])
            if case_id and judgment and summary:
                result[case_id] = {'document': judgment, 'summary': summary}
    return result

def tagged_text_from(row):
    text = first_value(row, ['tagged_judgment', 'tagged_text', 'tagged_document', 'text_with_tags'])
    if text:
        return text
    roles = row.get('sentence_roles')
    if isinstance(roles, list):
        lines = []
        for item in roles:
            if isinstance(item, dict):
                sent = first_value(item, ['sentence', 'text', 'sent_text'])
                role = first_value(item, ['role', 'label', 'rhetorical_role'])
                if sent:
                    lines.append(f'[{role or "Unclassified"}] {sent}')
        if lines:
            return '\n'.join(lines)
    return None

def read_tagged_jsonl(path):
    result = {}
    with path.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            case_id = norm_id(row.get('doc_id') or row.get('id') or row.get('file_id') or row.get('source_file'))
            if case_id:
                result[case_id] = tagged_text_from(row)
    return result

def add_roles_csv(tagged_by_id, csv_path):
    if csv_path is None:
        return
    roles_by_id = {}
    with csv_path.open(encoding='utf-8-sig', newline='') as f:
        for item in csv.DictReader(f):
            case_id = norm_id(item.get('doc_id') or item.get('id') or item.get('file_id'))
            sentence = first_value(item, ['sentence', 'text', 'sent_text'])
            role = first_value(item, ['role', 'label', 'rhetorical_role'])
            if case_id and sentence:
                roles_by_id.setdefault(case_id, []).append(f'[{role or "Unclassified"}] {sentence}')
    for case_id, lines in roles_by_id.items():
        if case_id in tagged_by_id and not tagged_by_id[case_id]:
            tagged_by_id[case_id] = '\n'.join(lines)
    print(f'IDs with roles in {csv_path.name}:', len(roles_by_id))

def match_records(original_by_id, tagged_by_id, split_label):
    rows = []
    missing_original = missing_tags = 0
    for case_id, tagged in tagged_by_id.items():
        original = original_by_id.get(case_id)
        if original is None:
            missing_original += 1
        elif not tagged:
            missing_tags += 1
        else:
            rows.append({'id': case_id, 'document': original['document'],
                         'tagged_document': tagged, 'summary': original['summary']})
    rows.sort(key=lambda x: x['id'])
    print(f'{split_label}: matched {len(rows)} pairs; no original pair {missing_original}; unusable tags {missing_tags}')
    if not rows:
        raise ValueError(f'No {split_label} pairs matched. Check document IDs and the selected input paths.')
    return rows

original_train = read_original_jsonl(TRAIN_JSONL)
original_test = read_original_jsonl(TEST_JSONL)
train_tags = read_tagged_jsonl(TRAIN_TAGGED_JSONL)
test_tags = read_tagged_jsonl(TEST_TAGGED_JSONL)
add_roles_csv(train_tags, TRAIN_ROLES_CSV)
add_roles_csv(test_tags, TEST_ROLES_CSV)

train_records = match_records(original_train, train_tags, 'Tagged training set')
test_records = match_records(original_test, test_tags, 'Tagged test set')
print('Example train ID:', train_records[0]['id'])
print('Train tagged preview:', train_records[0]['tagged_document'][:250])
print('Train summary preview:', train_records[0]['summary'][:250])
print('Example test ID:', test_records[0]['id'])

## 4. Split the 1,200 tagged training documents(1080 train/120 validation)


In [ ]:
import csv
from sklearn.model_selection import train_test_split

SEED = 42
MAX_SOURCE_TOKENS = 4096  # lower to 3072 if GPU memory runs out
MAX_TARGET_TOKENS = 512
CHUNK_TOKENS = 3500
CHUNK_OVERLAP = 300
MAX_EVAL_DOCS = None  # None evaluates all matched tagged test documents

train_ids, valid_ids = train_test_split(
    [r['id'] for r in train_records], test_size=0.10, random_state=SEED
)
train_by_id = {r['id']: r for r in train_records}
splits = {
    'train': [train_by_id[i] for i in train_ids],
    'validation': [train_by_id[i] for i in valid_ids],
    'test': test_records,
}
print({name: len(rows) for name, rows in splits.items()})
print('Train/test ID overlap:', bool(set(r['id'] for r in train_records) & set(r['id'] for r in test_records)))
OUTPUT_ROOT = Path('/kaggle/working/led_tagged_1200')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with (OUTPUT_ROOT / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id', 'split'])
    for split_name in ['train', 'validation']:
        for row in splits[split_name]:
            w.writerow([row['id'], split_name])
    for row in splits['test']:
        w.writerow([row['id'], 'external_test'])
print('Split manifest:', OUTPUT_ROOT / 'split_manifest.csv')

## 5. Configure LED and the LoRA adapter


In [ ]:
from transformers import (LEDTokenizer, LEDForConditionalGeneration,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer)
from peft import LoraConfig, TaskType, get_peft_model
from datasets import Dataset
import numpy as np

MODEL_NAME = 'allenai/led-base-16384'
OUTPUT_ROOT = Path('/kaggle/working/led_tagged_1200')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
tokenizer = LEDTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer ready. Kaggle GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
%pip uninstall -y torchao

## 6. Train Longformer Encoder Decoder Using Rhetorical Tagged judgements
The model architecture used in this project is **Longformer-Encoder-Decoder (LED)**, introduced by Beltagy, Peters, and Cohan in *Longformer: The Long-Document Transformer* (2020). LED adapts Longformer for long-document generation by combining an encoder with efficient local and global attention and a text-generating decoder. We use the pretrained `allenai/led-base-16384` checkpoint through Hugging Face Transformers and fine-tune it on our rhetorically tagged legal judgments using LoRA.

**Reference:** Beltagy, I., Peters, M. E., & Cohan, A. (2020). [Longformer: The Long-Document Transformer](https://arxiv.org/abs/2004.05150).


In [ ]:
def make_hf_dataset(rows, input_key):
    return Dataset.from_dict({
        'id': [r['id'] for r in rows],
        'source': [r[input_key] for r in rows],
        'summary': [r['summary'] for r in rows],
    })

def train_one(name, input_key):
    print(f'\\n===== Training {name} =====')
    model = LEDForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.max_length = MAX_TARGET_TOKENS
    lora = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32,
                      lora_dropout=0.1, target_modules=['q_proj', 'k_proj', 'v_proj'])
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    def preprocess(batch):
        enc = tokenizer(batch['source'], max_length=MAX_SOURCE_TOKENS,
                        truncation=True, padding=False)
        targets = tokenizer(text_target=batch['summary'], max_length=MAX_TARGET_TOKENS,
                             truncation=True, padding=False)
        enc['labels'] = targets['input_ids']
        enc['global_attention_mask'] = [[1] + [0] * (len(ids) - 1) for ids in enc['input_ids']]
        return enc

    tokenized = {}
    for split_name in ['train', 'validation']:
        ds = make_hf_dataset(splits[split_name], input_key)
        tokenized[split_name] = ds.map(preprocess, batched=True, remove_columns=ds.column_names)

    run_dir = OUTPUT_ROOT / name
    use_cuda = torch.cuda.is_available()
    use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=3,
        learning_rate=3e-4,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        fp16=bool(use_cuda and not use_bf16),
        bf16=use_bf16,
        predict_with_generate=False,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_steps=10,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        dataloader_num_workers=0,
        optim='adamw_torch',
        seed=SEED,
        data_seed=SEED,
    )
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8 if use_cuda else None)
    trainer = Seq2SeqTrainer(model=model, args=args,
        train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
        data_collator=collator, tokenizer=tokenizer)
    trainer.train()
    final_dir = run_dir / 'final'
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(final_dir)
    print('Saved adapter:', final_dir)
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return final_dir

tagged_dir = train_one('tagged', 'tagged_document')

## 7. Evaluate on the separate 100 tagged test judgments


In [ ]:
from pathlib import Path
tagged_dir = Path("/kaggle/working/led_tagged_1200/tagged/final")

In [ ]:
MODEL_NAME = "allenai/led-base-16384"

In [ ]:
%pip install -q bert-score

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as compute_bertscore
from transformers import (
    LEDForConditionalGeneration,
    BartTokenizer,
    BartForConditionalGeneration,
)
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


def chunk_text(text):
    token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]

    if len(token_ids) <= MAX_SOURCE_TOKENS:
        return [text]

    chunks = []
    start = 0
    step = max(1, CHUNK_TOKENS - CHUNK_OVERLAP)

    while start < len(token_ids):
        piece = token_ids[start:start + CHUNK_TOKENS]
        if not piece:
            break

        chunks.append(tokenizer.decode(piece, skip_special_tokens=True))

        if start + CHUNK_TOKENS >= len(token_ids):
            break

        start += step

    return chunks


@torch.inference_mode()
def summarize_chunked(model, source):
    chunk_summaries = []

    for chunk in chunk_text(source):
        enc = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_TOKENS,
        )

        enc["global_attention_mask"] = torch.zeros_like(enc["attention_mask"])
        enc["global_attention_mask"][:, 0] = 1
        enc = {k: v.to(model.device) for k, v in enc.items()}

        out = model.generate(
            **enc,
            max_new_tokens=MAX_TARGET_TOKENS,
            num_beams=2,
            no_repeat_ngram_size=3,
        )

        chunk_summaries.append(
            tokenizer.decode(out[0], skip_special_tokens=True)
        )

    combined = " ".join(s for s in chunk_summaries if s.strip())

    # Condense chunk summaries if their combined text exceeds LED's input limit.
    while len(tokenizer(combined, add_special_tokens=False)["input_ids"]) > MAX_SOURCE_TOKENS:
        ids = tokenizer(combined, add_special_tokens=False)["input_ids"]
        pieces = [
            tokenizer.decode(ids[i:i + CHUNK_TOKENS], skip_special_tokens=True)
            for i in range(0, len(ids), CHUNK_TOKENS)
        ]

        reduced = []
        for piece in pieces:
            enc = tokenizer(
                piece,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SOURCE_TOKENS,
            )
            enc["global_attention_mask"] = torch.zeros_like(enc["attention_mask"])
            enc["global_attention_mask"][:, 0] = 1
            enc = {k: v.to(model.device) for k, v in enc.items()}

            out = model.generate(
                **enc,
                max_new_tokens=MAX_TARGET_TOKENS,
                num_beams=2,
            )
            reduced.append(tokenizer.decode(out[0], skip_special_tokens=True))

        combined = " ".join(reduced)

    return combined


@torch.inference_mode()
def bartscore_hypothesis_to_reference(hypotheses, references, batch_size=1):
    """Average BART log-likelihood of each reference given its generated summary."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint = "facebook/bart-large-cnn"

    bart_tokenizer = BartTokenizer.from_pretrained(checkpoint)
    bart_model = BartForConditionalGeneration.from_pretrained(checkpoint).to(device)
    bart_model.eval()

    values = []
    pad_id = bart_tokenizer.pad_token_id

    for start in range(0, len(hypotheses), batch_size):
        batch_hyp = hypotheses[start:start + batch_size]
        batch_ref = references[start:start + batch_size]

        src = bart_tokenizer(
            batch_hyp,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )
        tgt = bart_tokenizer(
            batch_ref,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )

        input_ids = src["input_ids"].to(device)
        attention_mask = src["attention_mask"].to(device)
        labels = tgt["input_ids"].to(device)

        target_mask = labels.ne(pad_id)
        labels_for_model = labels.masked_fill(~target_mask, -100)

        logits = bart_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_for_model,
        ).logits

        safe_labels = labels.masked_fill(~target_mask, 0)
        token_log_probs = F.log_softmax(logits, dim=-1)
        token_log_probs = token_log_probs.gather(
            dim=-1,
            index=safe_labels.unsqueeze(-1),
        ).squeeze(-1)

        sequence_scores = (
            (token_log_probs * target_mask).sum(dim=1)
            / target_mask.sum(dim=1).clamp(min=1)
        )
        values.extend(sequence_scores.cpu().tolist())

    del bart_model, bart_tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return values


def evaluate_adapter(adapter_dir, input_key, label):
    base = LEDForConditionalGeneration.from_pretrained(MODEL_NAME)

    from peft import PeftModel
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model = model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    eval_rows = (
        splits["test"]
        if MAX_EVAL_DOCS is None
        else splits["test"][:MAX_EVAL_DOCS]
    )

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True,
    )

    rouge_scores = []
    predictions = []
    references = []

    for i, row in enumerate(eval_rows, start=1):
        prediction = summarize_chunked(model, row[input_key])
        reference = row["summary"]

        predictions.append(prediction)
        references.append(reference)
        rouge_scores.append(scorer.score(reference, prediction))

        if i % 5 == 0 or i == len(eval_rows):
            print(f"{label}: generated {i}/{len(eval_rows)} summaries")

    result = {
        metric: float(np.mean([score[metric].fmeasure for score in rouge_scores]))
        for metric in ["rouge1", "rouge2", "rougeL"]
    }

    # Free LED memory before loading the separate metric models.
    del model, base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # BARTScore: average log-likelihood of the reference given the prediction.
    bart_values = bartscore_hypothesis_to_reference(predictions, references)
    result["bartscore_hyp_to_ref"] = float(np.mean(bart_values))

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # BERTScore: contextual similarity between generated and reference summaries.
    device = "cuda" if torch.cuda.is_available() else "cpu"
    bert_precision, bert_recall, bert_f1 = compute_bertscore(
        predictions,
        references,
        lang="en",
        model_type="roberta-large",
        device=device,
        batch_size=4,
        verbose=True,
    )

    result["bertscore_precision"] = float(bert_precision.mean().cpu())
    result["bertscore_recall"] = float(bert_recall.mean().cpu())
    result["bertscore_f1"] = float(bert_f1.mean().cpu())

    results_table = pd.DataFrame([{"Model": label, **result}])
    display(results_table.round(4))

    return result


tagged_scores = evaluate_adapter(
    tagged_dir,
    "tagged_document",
    "Tagged LED",
)

print("Scores:", tagged_scores)

In [ ]:
from pathlib import Path
import json
import pandas as pd

save_dir = Path("/kaggle/working/led_tagged_1200")
save_dir.mkdir(parents=True, exist_ok=True)

with (save_dir / "tagged_led_metrics.json").open("w") as f:
    json.dump(tagged_scores, f, indent=2)

pd.DataFrame([{"Model": "Tagged LED", **tagged_scores}]).to_csv(
    save_dir / "tagged_led_metrics.csv", index=False
)

print("Saved scores to:", save_dir)

## 8. Save the split, metrics, and tagged model adapter


In [ ]:
import csv
with (OUTPUT_ROOT / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['id', 'split'])
    for split_name, rows in splits.items():
        for row in rows: w.writerow([row['id'], split_name])
with (OUTPUT_ROOT / 'tagged_test_metrics.json').open('w', encoding='utf-8') as f:
    json.dump({'model': 'tagged LED', 'tagged': tagged_scores,
               'max_source_tokens': MAX_SOURCE_TOKENS, 'chunk_tokens': CHUNK_TOKENS,
               'chunk_overlap': CHUNK_OVERLAP, 'seed': SEED}, f, indent=2)
print('Comparison artifacts saved in:', OUTPUT_ROOT)
print('Contents:', [p.name for p in OUTPUT_ROOT.iterdir()])